# 🧠 Semana 3 — Backpropagation y Funciones de Activación en Redes Neuronales

**Curso:** CADI Deep Learning  
**Objetivo:** Implementar y validar el proceso de aprendizaje de una red neuronal mediante **backpropagation** y el uso de **funciones de activación**, evidenciando cómo la red ajusta sus parámetros (pesos y sesgos) para reducir el error durante el entrenamiento.

---

## 📌 Definición

Una red neuronal aprende **ajustando sus pesos automáticamente** a partir de sus errores. Ese proceso se llama **backpropagation** (retropropagación). El flujo completo es:

```
1. Forward pass  →  la red hace una predicción
2. Cálculo de pérdida (loss)  →  se mide cuán equivocada estuvo
3. Backward pass  →  se calcula el gradiente del error respecto a cada peso
4. Actualización de parámetros  →  los pesos se ajustan para reducir el error
```

Además, compararemos dos **funciones de activación** clave:
- **Sigmoide** → convierte z en una probabilidad entre 0 y 1
- **ReLU** → deja pasar valores positivos y bloquea los negativos

Usaremos el problema **XOR** como dataset de clasificación binaria, ya que NO es linealmente separable y exige una red con capa oculta.

---
## 🔧 Parte 1 — Funciones de activación y sus derivadas

Las funciones de activación introducen **no linealidad** en la red, lo que le permite aprender patrones complejos. Para que backpropagation funcione, cada función debe tener una **derivada** (gradiente) que indique cómo cambia su salida.

| Función | Fórmula | Derivada | Rango |
|---|---|---|---|
| **Sigmoide** | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma'(a) = a \cdot (1 - a)$ | (0, 1) |
| **ReLU** | $f(z) = \max(0, z)$ | $f'(z) = 1$ si $z > 0$, si no $0$ | $[0, +\infty)$ |

> **Nota:** La derivada de Sigmoide se calcula sobre la **salida activada** `a`, no sobre `z`. La de ReLU es 1 donde `z > 0` y 0 en el resto.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ─────────────────────────────────────────────
# FUNCIONES DE ACTIVACIÓN y sus derivadas
# ─────────────────────────────────────────────

def sigmoid(x):
    """
    Convierte cualquier valor z en una probabilidad entre 0 y 1.
    Útil en la capa de salida para clasificación binaria.
    """
    return 1 / (1 + np.exp(-x))

def sigmoid_der(a):
    """
    Derivada de Sigmoide. Se calcula sobre la salida activada 'a',
    no sobre z. Usada en backpropagation para propagar el gradiente.
    """
    return a * (1 - a)

def relu(x):
    """
    Deja pasar valores positivos sin cambio; convierte negativos en 0.
    Evita el problema de gradiente desvanecido en capas ocultas.
    """
    return np.maximum(0, x)   # np.maximum opera elemento a elemento (distinto de np.max)

def relu_der(a):
    """
    Derivada de ReLU: 1 donde la salida activada > 0, 0 en el resto.
    """
    return (a > 0).astype(float)


# ─────────────────────────────────────────────
# VISUALIZACIÓN: comparación de ambas funciones
# ─────────────────────────────────────────────
z_vals = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Funciones de Activación", fontsize=14, fontweight='bold')

# Sigmoide
axes[0].plot(z_vals, sigmoid(z_vals), color='#2196F3', linewidth=2.5, label='Sigmoide')
axes[0].plot(z_vals, sigmoid_der(sigmoid(z_vals)), color='#2196F3', linewidth=2, linestyle='--', label='Derivada')
axes[0].set_title("Sigmoide")
axes[0].set_xlabel("z")
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle=':')
axes[0].axvline(0, color='gray', linewidth=0.8, linestyle=':')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ReLU
axes[1].plot(z_vals, relu(z_vals), color='#FF5722', linewidth=2.5, label='ReLU')
axes[1].plot(z_vals, relu_der(relu(z_vals)), color='#FF5722', linewidth=2, linestyle='--', label='Derivada')
axes[1].set_title("ReLU")
axes[1].set_xlabel("z")
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle=':')
axes[1].axvline(0, color='gray', linewidth=0.8, linestyle=':')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Funciones de activación y sus derivadas definidas correctamente.")

---
## 🏗️ Parte 2 — Arquitectura de la Red Neuronal con Backpropagation

Vamos a construir una red neuronal **configurable** que soporte:
- Cualquier número de capas y neuronas (definidas como lista)
- Selección de función de activación: `sigmoid` o `relu`
- Entrenamiento completo con **forward pass**, **cálculo de pérdida (MSE)** y **backward pass**

### Arquitectura del experimento:
```
Entrada [2 neuronas]  →  Capa oculta [4 neuronas]  →  Salida [1 neurona]
```

Los pesos se inicializan con valores pequeños aleatorios (`× 0.1`) para evitar que las activaciones se saturen desde el inicio.

In [ ]:
# ─────────────────────────────────────────────
# RED NEURONAL CON BACKPROPAGATION
# ─────────────────────────────────────────────

class RedNeuronal:
    """
    Red neuronal totalmente conectada con backpropagation.

    Parámetros:
      capas      : lista con el número de neuronas por capa
                   Ejemplo: [2, 4, 1] → 2 entradas, 4 ocultas, 1 salida
      activacion : 'sigmoid' o 'relu'
    """

    def __init__(self, capas, activacion='sigmoid'):
        self.capas = capas
        self.activacion_nombre = activacion
        self.loss_history = []

        # Inicialización de pesos y sesgos para cada par de capas consecutivas
        # Pesos pequeños (×0.1) para evitar saturación inicial de las activaciones
        self.W = [np.random.randn(capas[i], capas[i+1]) * 0.1
                  for i in range(len(capas) - 1)]
        self.b = [np.zeros((1, capas[i+1]))
                  for i in range(len(capas) - 1)]

    # ── Función de activación seleccionada ──
    def _activacion(self, x):
        if self.activacion_nombre == 'sigmoid':
            return sigmoid(x)
        return relu(x)

    # ── Derivada de la función de activación ──
    def _activacion_der(self, a):
        if self.activacion_nombre == 'sigmoid':
            return sigmoid_der(a)
        return relu_der(a)

    # ──────────────────────────────────────────
    # FORWARD PASS: propagación hacia adelante
    # Calcula la activación de cada capa en orden
    # ──────────────────────────────────────────
    def forward(self, X):
        """
        Propaga la entrada X a través de todas las capas.
        Guarda cada activación en self.A para usarlas en backpropagation.
        """
        self.A = [X]   # self.A[0] = entradas originales
        for i in range(len(self.W)):
            Z = np.dot(self.A[-1], self.W[i]) + self.b[i]   # suma ponderada
            self.A.append(self._activacion(Z))               # activación
        return self.A[-1]   # última activación = predicción final

    # ──────────────────────────────────────────
    # ENTRENAMIENTO: forward + pérdida + backward + actualización
    # ──────────────────────────────────────────
    def train(self, X, y, epochs=2000, lr=0.1):
        """
        Entrena la red durante un número de épocas.

        Parámetros:
          X      : entradas del dataset
          y      : etiquetas correctas
          epochs : número de iteraciones de entrenamiento
          lr     : tasa de aprendizaje (learning rate)
        """
        for epoch in range(epochs):

            # 1. FORWARD PASS: obtener predicción actual
            y_pred = self.forward(X)

            # 2. PÉRDIDA (MSE): medir cuán equivocada está la red
            #    error = y_real - y_predicho (con signo, para saber la dirección)
            error = y - y_pred
            if epoch % 100 == 0:
                mse = np.mean(np.square(error))
                self.loss_history.append(mse)

            # 3. BACKWARD PASS: calcular deltas capa por capa (de salida a entrada)
            #    delta_salida = error × derivada de la activación en la capa de salida
            deltas = [error * self._activacion_der(self.A[-1])]

            # Propagar el gradiente hacia las capas anteriores
            for i in reversed(range(len(self.W) - 1)):
                # delta_capa_i = (delta_capa_i+1 × pesos_i+1^T) × derivada_activación_i
                delta = deltas[-1].dot(self.W[i+1].T) * self._activacion_der(self.A[i+1])
                deltas.append(delta)
            deltas.reverse()   # ordenar de entrada a salida

            # 4. ACTUALIZACIÓN DE PARÁMETROS (Descenso de Gradiente)
            #    w_nuevo = w_actual + lr × (activación_anterior^T · delta)
            for i in range(len(self.W)):
                self.W[i] += self.A[i].T.dot(deltas[i]) * lr
                self.b[i] += np.sum(deltas[i], axis=0, keepdims=True) * lr


print("✅ Clase RedNeuronal definida correctamente.")
print("   Soporta: forward pass, cálculo MSE, backpropagation y descenso de gradiente.")

---
## 🧪 Parte 3 — Dataset y Entrenamiento Comparativo

### ¿Por qué XOR?
El problema XOR (OR exclusivo) NO es linealmente separable: ninguna línea recta puede dividir correctamente los cuatro casos. Por eso es el ejemplo clásico para probar redes con capas ocultas.

| x1 | x2 | XOR (y) |
|----|----|---------|
| 0  | 0  | 0       |
| 0  | 1  | 1       |
| 1  | 0  | 1       |
| 1  | 1  | 0       |

Entrenaremos **dos redes con la misma arquitectura** `[2, 4, 1]` pero con funciones de activación diferentes para comparar su comportamiento.

In [ ]:
# ─────────────────────────────────────────────
# DATASET: problema XOR
# ─────────────────────────────────────────────
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([[0],   # 0 XOR 0 = 0
              [1],   # 0 XOR 1 = 1
              [1],   # 1 XOR 0 = 1
              [0]])  # 1 XOR 1 = 0

print("Dataset XOR:")
df_xor = pd.DataFrame(
    np.hstack([X, y]),
    columns=["x1", "x2", "y (XOR)"]
).astype(int)
print(df_xor.to_string(index=False))


# ─────────────────────────────────────────────
# ENTRENAMIENTO: Sigmoid vs ReLU
# Fijamos la misma semilla para que los pesos iniciales sean idénticos
# y la comparación sea justa
# ─────────────────────────────────────────────
np.random.seed(42)
nn_sigmoid = RedNeuronal([2, 4, 1], activacion='sigmoid')

np.random.seed(42)
nn_relu = RedNeuronal([2, 4, 1], activacion='relu')

print("\n⏳ Entrenando red con Sigmoide (20,000 épocas)...")
nn_sigmoid.train(X, y, epochs=20000, lr=0.5)

print("⏳ Entrenando red con ReLU (20,000 épocas)...")
nn_relu.train(X, y, epochs=20000, lr=0.3)

print("\n✅ Entrenamiento completado.")

---
## 📊 Parte 4 — Métricas y Resultados

Evaluamos cada modelo con:
- **Loss final (MSE):** qué tan cerca están las predicciones de los valores reales
- **Accuracy:** porcentaje de casos clasificados correctamente
- **Predicciones caso a caso:** verificamos cada entrada individualmente

In [ ]:
# ─────────────────────────────────────────────
# GRÁFICA: curva de pérdida durante el entrenamiento
# ─────────────────────────────────────────────
plt.figure(figsize=(10, 5))
plt.plot(nn_sigmoid.loss_history, label='Sigmoide', color='#2196F3', linewidth=2)
plt.plot(nn_relu.loss_history,    label='ReLU',     color='#FF5722', linewidth=2)
plt.title('Comparación de Funciones de Activación — Pérdida (MSE)', fontsize=13, fontweight='bold')
plt.xlabel('Iteraciones (×100 épocas)')
plt.ylabel('Error Cuadrático Medio (MSE)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────
# MÉTRICAS FINALES: accuracy y loss
# ─────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════╗")
print("║              MÉTRICAS FINALES DE ENTRENAMIENTO       ║")
print("╚══════════════════════════════════════════════════════╝")

for nombre, modelo in [('Sigmoid', nn_sigmoid), ('ReLU', nn_relu)]:
    pred       = modelo.forward(X)
    pred_clase = (pred >= 0.5).astype(int)
    acc        = np.mean(pred_clase == y) * 100
    loss_final = modelo.loss_history[-1]
    print(f"  {nombre:8s} | Loss final: {loss_final:.6f} | Accuracy: {acc:.1f}%")


# ─────────────────────────────────────────────
# TABLA DETALLADA: predicciones caso a caso
# ─────────────────────────────────────────────
print("\n╔══════════════════════════════════════════════════════════════════════╗")
print("║                  PREDICCIONES CASO A CASO                           ║")
print("╚══════════════════════════════════════════════════════════════════════╝")

pred_sig  = nn_sigmoid.forward(X)
pred_relu = nn_relu.forward(X)
clase_sig  = (pred_sig  >= 0.5).astype(int)
clase_relu = (pred_relu >= 0.5).astype(int)

filas = []
for i in range(len(X)):
    filas.append({
        "x1": int(X[i, 0]),
        "x2": int(X[i, 1]),
        "y (real)": int(y[i, 0]),
        "Prob Sigmoid": round(float(pred_sig[i, 0]),  4),
        "ŷ Sigmoid":   int(clase_sig[i, 0]),
        "✔ Sig": "✅" if clase_sig[i, 0]  == y[i, 0] else "❌",
        "Prob ReLU":  round(float(pred_relu[i, 0]), 4),
        "ŷ ReLU":     int(clase_relu[i, 0]),
        "✔ ReLU": "✅" if clase_relu[i, 0] == y[i, 0] else "❌",
    })

print(pd.DataFrame(filas).to_string(index=False))

---
## 📝 Parte 5 — Análisis y Conclusiones

### 5.1 ¿Cómo funciona el backpropagation?

Backpropagation calcula el **gradiente del error** respecto a cada peso, recorriendo la red de salida a entrada. Para cada capa:

1. Se calcula el **delta** (error ponderado por la derivada de la activación)
2. Se propaga hacia la capa anterior multiplicando por la transpuesta de los pesos
3. Los pesos se actualizan sumando `lr × activación_anterior^T × delta`

Esto garantiza que cada peso sea ajustado **proporcionalmente a su responsabilidad en el error**.

---

### 5.2 Sigmoide vs ReLU: ¿cuál converge mejor?

| Aspecto | Sigmoide | ReLU |
|---------|----------|------|
| **Rango de salida** | (0, 1) — probabilidades | [0, ∞) — valores positivos |
| **Derivada máxima** | 0.25 (en z=0) | 1 (para z > 0) |
| **Problema** | Gradiente desvanecido en redes profundas | Neuronas muertas si z < 0 siempre |
| **Convergencia en XOR** | Más lenta pero estable | Más rápida con lr adecuada |
| **Uso típico** | Capa de salida binaria | Capas ocultas |

En la gráfica de pérdida se puede observar que **ambas convergen al 100% de accuracy**, pero sus curvas de descenso del error tienen formas distintas: Sigmoide tiende a descender de forma más suave, mientras que ReLU puede tener caídas más abruptas.

---

### 5.3 Efecto de la tasa de aprendizaje (lr)

- **lr demasiado alta:** los pesos oscilan y la red puede no converger.
- **lr demasiado baja:** la red converge, pero tarda muchas más épocas.
- En este experimento se usó `lr=0.5` para Sigmoide y `lr=0.3` para ReLU, ajustando según la magnitud de los gradientes que genera cada activación.

---

### 5.4 Conclusión principal

Backpropagation permite que la red **aprenda de sus errores iteración a iteración**, ajustando pesos y sesgos de forma automática. La función de activación influye directamente en la **velocidad de convergencia** y en la **estabilidad del gradiente**. Para arquitecturas como la de este experimento (`[2, 4, 1]`), ambas funciones logran resolver el problema XOR, aunque con dinámicas de entrenamiento diferentes.